# Surena × LIBERO Environment

## 1. Setup

In [ ]:
# Imports, paths, and global runtime setup
from __future__ import annotations

import os, sys, time, gc, math, traceback
from pathlib import Path

# Project paths (override with environment variables when needed)
SURENA_WORKSPACE = Path(os.environ.get("SURENA_WORKSPACE", Path.home())).expanduser().resolve()
SURENA_REPO_ROOT = Path(os.environ.get(
    "SURENA_VLA_PROJECT_ROOT", SURENA_WORKSPACE / "surena-vla-manipulation"
)).expanduser().resolve()
OPENVLA_ROOT = Path(os.environ.get(
    "OPENVLA_ROOT", SURENA_WORKSPACE / "openvla-mini"
)).expanduser().resolve()
LIBERO_DIR = Path(os.environ.get(
    "LIBERO_ROOT", SURENA_WORKSPACE / "LIBERO"
)).expanduser().resolve()

# VLA checkpoints
CHECKPOINT_ROOT = Path(os.environ.get(
    "SURENA_VLA_CHECKPOINT_ROOT",
    "/media/parsa/OS/Users/parsa/Desktop/vla/checkpoints",
)).expanduser().resolve()

VLA_CHECKPOINT_PLAIN = (
    CHECKPOINT_ROOT / "minivla-libero90/checkpoints/step-122500-epoch-55-loss=0.0743.pt"
)
VLA_CHECKPOINT_VQ = (
    CHECKPOINT_ROOT / "minivla-libero90-vq/checkpoints/step-150000-epoch-67-loss=0.0934.pt"
)

VLA_CHECKPOINT = VLA_CHECKPOINT_PLAIN
# VLA_CHECKPOINT = VLA_CHECKPOINT_VQ

VLA_UNNORM_KEY = "libero_90"

# VQ tokenizer folder
VQ_NAME = "pretrain_vq+mx-libero_90+fach-7+ng-7+nemb-128+nlatent-512"
VQ_DIR  = OPENVLA_ROOT / "vq" / VQ_NAME
REQUIRE_VQ_FILES = bool(VLA_CHECKPOINT == VLA_CHECKPOINT_VQ)

# Default Surena/VLA bridge parameters
IK_MAX_REACH = 0.70
HONEST_STICKY_ATTACH_DISTANCE = 0.07


# Environment variables
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PRISMATIC_DATA_ROOT", "/tmp/prismatic_data")
os.environ.setdefault("HF_HOME", str(Path(os.environ.get("SURENA_VLA_HF_HOME", "/media/parsa/OS/Users/parsa/Desktop/vla/hf_cache")).expanduser()))

# Avoid TensorFlow/JAX side imports from transformers/prismatic.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# All three repositories are installed in editable mode; no sys.path mutation is needed.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from PIL import Image

import mujoco
import mink

import importlib

from surena_vla.control import (
    SurenaArmController,
    HingeContactGuidance,
    StickyGripper,
    HOME_QPOS,
    DEG,
    MAX_REACH,
    ARM_INDICES,
    GAZEBO_INDEX_MAP_BARE,
    clamp_joints,
    clamp_target_lag,
    get_loop_params,
    render_frame,
    reset_settle_rebind,
    execute_joint_target,
    make_rgb_animation,
    save_episode_video,
)
from surena_vla.paths import OUTPUTS_ROOT, VIDEOS_ROOT
from surena_vla.telemetry import (
    EpisodeLogger,
    collect_session_meta,
    sample_memory_usage,
)

RUNS_ROOT = Path(os.environ.get(
    "SURENA_VLA_RUNS_ROOT", OUTPUTS_ROOT / "runs"
)).expanduser().resolve()

plt.rcParams["animation.embed_limit"] = 80  # MB
print("Imports OK")


## 2. Load VLA model

In [ ]:
# Load VLA model  (GPU, run once)

import torch
import psutil
from prismatic.models.load import load_vla

if REQUIRE_VQ_FILES:
    os.chdir(OPENVLA_ROOT)

_orig_torch_load = torch.load
def _patched_torch_load(f, map_location=None, **kwargs):
    if map_location is None:
        map_location = "cpu"
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(f, map_location=map_location, **kwargs)

torch.load = _patched_torch_load

print(f"RAM free:  {psutil.virtual_memory().available / 1e9:.1f} GB")
if torch.cuda.is_available():
    free_vram, total_vram = torch.cuda.mem_get_info()
    print(f"GPU:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {free_vram / 1e9:.2f} / {total_vram / 1e9:.2f} GB")
else:
    raise RuntimeError("CUDA is not available. This notebook expects GPU inference.")

gc.collect()
torch.cuda.empty_cache()

t0 = time.time()
print("Loading VLA...")

vla = load_vla(VLA_CHECKPOINT, hf_token=None, load_for_training=False)
vla = vla.to("cuda", dtype=torch.float16).eval()
vla.enable_mixed_precision_training = False

gc.collect()
torch.cuda.empty_cache()

print(f"Loaded in {time.time() - t0:.1f} s")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Session-level telemetry metadata (hardware, package versions, git commit,
# checkpoint identity). Collected once per kernel session and duplicated into
# every episode's .h5 file so each rollout stays self-contained.
SESSION_META = collect_session_meta(vla_checkpoint=VLA_CHECKPOINT, vla_unnorm_key=VLA_UNNORM_KEY)
print("\nSession id:", SESSION_META["session_id"])
print("GPU:", SESSION_META.get("gpu_name"), "| VRAM total:", SESSION_META.get("vram_total_gb"), "GB")
print("git_commit:", SESSION_META.get("git_commit"), "| dirty:", SESSION_META.get("git_dirty"))


## 3. LIBERO Task Environment Creation

In [ ]:
# LIBERO environment creation + episode presets

from surena_vla.integrations import register_all
register_all()

from libero.libero.envs.bddl_base_domain import TASK_MAPPING
import surena_vla.integrations.libero.surena_manipulation as sm

importlib.reload(sm)

for k in list(TASK_MAPPING.keys()):
    if "surena" in k.lower():
        TASK_MAPPING.pop(k)

sm.register_surena_tasks(TASK_MAPPING)

joint_pos_config = {
    "type": "JOINT_POSITION",
    "interpolation": None,
    "ramp_ratio": 0.2,
    "input_max": 1, 
    "input_min": -1,
    "output_max": 1.0, 
    "output_min": -1.0,
    "kp": 200, 
    "damping_ratio": 1,
    "impedance_mode": "fixed",
    "kp_limits": [0, 300], 
    "damping_ratio_limits": [0, 15],
    "position_limits": None, 
    "velocity_limits": None,
}

AVAILABLE_TASKS = sm.list_presets(verbose=False)

# TASK_PRESET = "close_top_drawer"
# TASK_PRESET = "close_microwave"
# TASK_PRESET = "put_ketchup_in_top_drawer"
TASK_PRESET = "put_black_bowl_on_plate"
# TASK_PRESET = "put_moka_pot_on_stove"
# TASK_PRESET = "put_book_in_left_caddy"
# TASK_PRESET = "turn_on_stove"
# TASK_PRESET = "put_frying_pan_on_stove"

# TASK_PRESET = "open_top_drawer"
# TASK_PRESET = "close_bottom_drawer"
# TASK_PRESET = "put_black_bowl_in_top_drawer"
# TASK_PRESET = "open_microwave"
# TASK_PRESET = "put_middle_black_bowl_on_plate"
# TASK_PRESET = "stack_middle_bowl_on_back_bowl"
# TASK_PRESET = "put_book_on_shelf"

cfg = sm.get_preset(TASK_PRESET)

# notebook-side overrides
cfg.update({
    # "max_vla_steps": 125,
    # "pos_scale": 0.035,
    # "rot_scale": 0.005,
    # "max_rot_delta": 0.020,
    # "sticky": False,
    # "sticky_attach_distance": 0.065,
    # "success_delta": 0.12,
    # "object_name_filter": None,
    "use_native_success": True,
})


if "env" in globals():
    try:
        env.close()
    except Exception:
        pass

env, instruction = sm.make_env(
    TASK_PRESET,
    controller_configs=joint_pos_config,
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    camera_names=["agentview"],
    camera_heights=224,
    camera_widths=224,
    control_freq=20,
    horizon=max(500, int(cfg.get("max_vla_steps", 125)) * 50),
    reward_shaping=False,
    debug_reachability=False,
    warn_missing_scene_elements=False,
    use_libero_success_first=bool(cfg.get("use_libero_success_first", True)),
)

obs = env.reset()


print("TASK_PRESET:", TASK_PRESET)
print("Instruction for VLA:", repr(instruction))
print("mode:", cfg["mode"])
print("max_vla_steps:", cfg.get("max_vla_steps"))
print("pos_scale:", cfg.get("pos_scale"))
print("rot_scale:", cfg.get("rot_scale"))
print("sticky:", cfg.get("sticky"))
print("sticky_attach_distance:", cfg.get("sticky_attach_distance"))
print("robot0_eef_pos:", np.round(obs["robot0_eef_pos"], 4))
print("action_dim:", env.action_dim)


plt.figure(figsize=(8, 5))
img = render_frame(env, camera_name="agentview")
plt.imshow(img)
plt.axis("off")
plt.show()

## 4. Controller preparation

In [ ]:
# Create / rebind controller and apply the recommended dynamic execution settings

def prepare_surena_controller(
    env,
    ctrl=None,
    *,
    settle_steps=50,
    pos_scale=0.035,
    rot_scale=0.003,
    max_rot_delta=0.01,
    sticky=False,
    sticky_attach_distance=HONEST_STICKY_ATTACH_DISTANCE,
    verbose=True,
):
    obs, ctrl = reset_settle_rebind(env, ctrl=ctrl, settle_steps=settle_steps)

    ctrl.configure_vla_execution(
        kp_major=1200.0,
        kp_wrist=600.0,
        damping_major=45.0,
        damping_wrist=25.0,
        gravity_comp=True,
        gravity_strength=1.0,
        verbose=verbose,
    )

    ctrl.vla.pos_scale = float(pos_scale)
    ctrl.vla.rot_scale = float(rot_scale)
    ctrl.vla.MAX_ROT_DELTA = float(max_rot_delta)
    ctrl.reset_vla()

    if sticky:
        # No object_name_filter here. The palm may attach to the nearest freejoint
        # object only when the gripper command is explicitly > 0.5.
        ctrl.enable_sticky_gripper(
            env=env,
            object_name_filter=None,
            attach_distance=sticky_attach_distance,
            close_threshold=0.5,
            release_threshold=0.5,
            verbose=False,
        )
    else:
        ctrl.disable_sticky_gripper()

    p, _ = ctrl.get_eef_pose()
    print("\nController ready.")
    print("EEF:", np.round(p, 6))
    print("pos_scale:", ctrl.vla.pos_scale)
    print("rot_scale:", ctrl.vla.rot_scale)
    print("sticky:", ctrl.sticky is not None)

    if ctrl.sticky is not None:
        print("sticky attach_distance:", ctrl.sticky.attach_distance)
        print("nearest freejoint candidates:")
        ctrl.print_gripper_candidates(max_rows=12)

    return obs, ctrl

## 5. Centralized VLA utilities

Live view, VLA inference, raw MuJoCo access, sticky-state logging, and ...

In [ ]:
# Centralized VLA episode utilities

try:
    import cv2
except Exception:
    cv2 = None

import types


def show_live_frame(frame, lines=None, window_name="Surena VLA live", enabled=True):
    if not enabled:
        return True

    if cv2 is None:
        try:
            env.render()
        except Exception:
            pass
        return True

    img = np.asarray(frame).copy()
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    if lines:
        y = 22
        for line in lines:
            cv2.putText(
                img,
                str(line),
                (8, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.48,
                (255, 255, 255),
                1,
                cv2.LINE_AA,
            )
            y += 20

    cv2.imshow(window_name, img)
    key = cv2.waitKey(1)
    return key != 27


def close_live_viewer():
    if cv2 is not None:
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass


def get_raw_model_data(env):
    model = env.sim.model._model if hasattr(env.sim.model, "_model") else env.sim.model
    data = env.sim.data._data if hasattr(env.sim.data, "_data") else env.sim.data
    return model, data


def predict_vla_action(frame, instruction):
    if "vla" not in globals():
        raise RuntimeError("Load the VLA model first.")

    image = Image.fromarray(np.asarray(frame).astype(np.uint8))

    with torch.no_grad():
        action = vla.predict_action(
            image=image,
            instruction=instruction,
            unnorm_key=VLA_UNNORM_KEY,
            do_sample=False,
        )

    return np.asarray(action, dtype=float)

def sticky_state(ctrl):
    if getattr(ctrl, "sticky", None) is None:
        return {"attached": False, "body_name": None}

    return {
        "attached": bool(ctrl.sticky.attached),
        "body_name": ctrl.sticky.attached_body_name,
    }

## 6. General episode runners

The important rule is simple: task differences live in `cfg` and in small action / success adapters. The main loop stays shared.

### 6.1 Episode configuration and success monitoring

In [ ]:
# Episode runner configuration + success monitoring

RUNNER_DEFAULTS = {
    "ctrl_ticks_per_vla_action": 35,
    "render_stride": 8,
    "verbose_every": 1,
    "soft_ik_warn_err": 0.055,
    "hard_ik_stop_err": 0.120,
    "rot_fraction_drawer": 0.50,
    "rot_fraction_articulation": 0.50,
    "rot_fraction_pick_place": 0.50,
    "grip_close_threshold": 0.50,
    "max_target_lag": 0.015,  # generic Cartesian anti-windup guard
    # Residual/blended grasp-assist guidance: per-phase blend weight between
    # raw VLA action (alpha=0) and the geometric guidance correction
    # (alpha=1). The VLA action is never discarded; guidance only nudges it.
    "grasp_blend_alpha_hover": 0.35,
    "grasp_blend_alpha_approach": 0.45,
    "grasp_blend_alpha_transport": 0.55,
    "grasp_blend_alpha_lower": 0.55,
    "grasp_blend_alpha_close": 0.15,  # close/settle stays close to deterministic
    "grasp_position_guidance_gain": 6.0,
    "grasp_rotation_guidance_gain": 3.0,
    "grasp_max_position_correction": 0.03,  # raw-action units (pre pos_scale)
    "grasp_max_rotation_correction": 0.30,  # raw-action units (pre rot_scale)
}


def resolve_episode_cfg(cfg_or_name=None):
    """Return a fresh notebook-side episode config dict."""
    if cfg_or_name is None:
        if "cfg" not in globals():
            raise RuntimeError("No cfg found. Run the LIBERO task environment creation cell first.")
        cfg_or_name = cfg

    if isinstance(cfg_or_name, str):
        out = sm.get_preset(cfg_or_name)
    else:
        out = dict(cfg_or_name)

    if "preset_name" not in out and "TASK_PRESET" in globals():
        out["preset_name"] = TASK_PRESET

    mode = out.get("mode", "pick_place")
    if mode not in {"drawer", "articulation", "pick_place", "combo", "drawer_sequence"}:
        raise ValueError(f"Unsupported episode mode: {mode!r}")

    for key, value in RUNNER_DEFAULTS.items():
        out.setdefault(key, value)

    out.setdefault("instruction", globals().get("instruction", ""))
    out.setdefault("max_vla_steps", 125)
    out.setdefault("pos_scale", 0.035)
    out.setdefault("rot_scale", 0.005)
    out.setdefault("max_rot_delta", 0.020)
    out.setdefault("sticky", mode in {"pick_place", "combo"})
    out.setdefault("sticky_attach_distance", HONEST_STICKY_ATTACH_DISTANCE)
    return out


def apply_episode_control_cfg(ctrl, cfg):
    """Apply VLA integration scales to the already prepared controller."""
    ctrl.vla.pos_scale = float(cfg["pos_scale"])
    ctrl.vla.rot_scale = float(cfg["rot_scale"])
    ctrl.vla.MAX_ROT_DELTA = float(cfg["max_rot_delta"])
    return ctrl


def configure_sticky_for_episode(env, ctrl, cfg):
    """Enable sticky only for modes/configs that explicitly need grasping."""
    if bool(cfg.get("sticky", False)):
        ctrl.enable_sticky_gripper(
            env=env,
            object_name_filter=cfg.get("object_name_filter", None),
            attach_distance=float(cfg.get("sticky_attach_distance", HONEST_STICKY_ATTACH_DISTANCE)),
            close_threshold=float(cfg.get("grip_close_threshold", 0.65)),
            release_threshold=float(cfg.get("grip_release_threshold", 0.35)),
            close_dwell_ticks=int(cfg.get("grip_close_dwell_ticks", 2)),
            release_dwell_ticks=int(cfg.get("grip_release_dwell_ticks", 2)),
            candidate_dwell_ticks=int(cfg.get("grip_candidate_dwell_ticks", 2)),
            verbose=False,
        )
        print("sticky=True")
        print("object_name_filter:", cfg.get("object_name_filter", None))
        print("sticky attach_distance:", ctrl.sticky.attach_distance)
        print("nearest freejoint candidates:")
        ctrl.print_gripper_candidates(max_rows=12)
    else:
        ctrl.disable_sticky_gripper()
        print("sticky=False")


def _resolve_mujoco_name(model, candidates, kind):
    if isinstance(candidates, str):
        candidates = [candidates]
    obj_type = {
        "joint": mujoco.mjtObj.mjOBJ_JOINT,
        "body": mujoco.mjtObj.mjOBJ_BODY,
        "site": mujoco.mjtObj.mjOBJ_SITE,
    }[kind]
    for name in candidates:
        if mujoco.mj_name2id(model, obj_type, name) >= 0:
            return name
    return None


def _collect_scalar_success_specs(spec):
    """Collect drawer/articulation specs from SUCCESS_SPEC, including nested all/any."""
    if not spec:
        return []
    typ = spec.get("type")
    if typ in {"drawer", "articulation"}:
        return [spec]
    if typ in {"all", "any"}:
        out = []
        for item in spec.get("items", []):
            out.extend(_collect_scalar_success_specs(item))
        return out
    return []


def make_episode_success_monitor(env, cfg, ctrl):
    """
    General success monitor.

    Primary signal: env._check_success(), which is implemented in surena_manipulation.
    Extra logging: scalar joint progress for drawer / articulation tasks.
    """
    model, data = get_raw_model_data(env)
    success_spec = getattr(env, "SUCCESS_SPEC", None)
    scalar_specs = _collect_scalar_success_specs(success_spec)

    joint_rows = []
    for spec in scalar_specs:
        name = _resolve_mujoco_name(model, spec.get("joint_names", []), "joint")
        if name is None:
            continue
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
        qadr = int(model.jnt_qposadr[jid])
        joint_rows.append({
            "name": name,
            "qadr": qadr,
            "type": spec.get("type"),
            "target": spec.get("target"),
            "q0": float(data.qpos[qadr]),
        })

    if joint_rows:
        print("[Success monitor] scalar joints:")
        for r in joint_rows:
            print(f"  {r['name']} | q0={r['q0']:+.4f} | target={r['target']}")
    else:
        print("[Success monitor] using env._check_success() only")

    def check(verbose=False):
        _, data_now = get_raw_model_data(env)

        use_native_success = bool(cfg.get("use_native_success", cfg.get("mode") not in {"drawer", "articulation"}))
        try:
            native_success = bool(env._check_success()) if use_native_success else False
        except Exception as exc:
            native_success = False
            native_error = repr(exc)
        else:
            native_error = None

        joint_info = []
        for r in joint_rows:
            q = float(data_now.qpos[r["qadr"]])
            delta = q - r["q0"]
            target = r["target"]

            target_met = False
            success_delta = cfg.get("success_delta", None)

            if r["type"] == "drawer":
                if success_delta is not None:
                    success_delta = abs(float(success_delta))
                    if target == "closed":
                        # Drawer closing means q moves upward toward 0.
                        target_met = delta >= success_delta
                    elif target == "open":
                        # Drawer opening means q moves downward / more negative.
                        target_met = -delta >= success_delta
                else:
                    # Fallback to surena_manipulation thresholds.
                    if target == "open":
                        target_met = q <= -0.08
                    elif target == "closed":
                        target_met = q >= -0.04
            elif r["type"] == "articulation":
                if target == "on":
                    target_met = q >= float(cfg.get("success_q_on", 0.45))
                elif target == "off":
                    target_met = q <= float(cfg.get("success_q_off", 0.25))
                elif target == "open":
                    target_met = q <= float(cfg.get("success_q_open", -0.50))
                elif target == "closed":
                    target_met = q >= float(cfg.get("success_q_closed", -0.25))

            joint_info.append({
                "name": r["name"],
                "q": q,
                "q0": r["q0"],
                "delta": delta,
                "target": target,
                "target_met": bool(target_met),
            })

        scalar_success = (
            cfg.get("mode") in {"drawer", "articulation"}
            and bool(joint_info)
            and all(item["target_met"] for item in joint_info)
        )
        currently_attached = bool(ctrl.sticky is not None and ctrl.sticky.attached)
        if currently_attached:
            cfg["_grasp_ever_attached"] = True
        ever_attached = bool(cfg.get("_grasp_ever_attached", False))
        release_started = bool(cfg.get("_grasp_release_started", False))
        placement_verified = bool(
            native_success and ever_attached and release_started and not currently_attached
        )
        assisted_pick_place = (
            cfg.get("mode") in {"pick_place", "combo"}
            and cfg.get("grasp_assist_body") is not None
        )
        success = placement_verified if assisted_pick_place else bool(native_success or scalar_success)
        info = {
            "success": success,
            "native_success": native_success,
            "native_error": native_error,
            "scalar_success": scalar_success,
            "ever_attached": ever_attached,
            "release_started": release_started,
            "currently_attached": currently_attached,
            "placement_verified": placement_verified,
            "joints": joint_info,
        }

        if verbose:
            if joint_info:
                best = max(joint_info, key=lambda x: abs(x["delta"]))
                print(
                    f"[success] {success} | native={native_success} | "
                    f"joint={best['name']} | q0={best['q0']:+.4f} | "
                    f"q={best['q']:+.4f} | delta={best['delta']:+.4f} | "
                    f"target={best['target']} | target_met={best['target_met']}"
                )
            else:
                print(f"[success] {success} | native={native_success}")

        return success, info

    return check


### 6.2 Action adapters

In [ ]:
# General VLA action adapter

def transform_vla_action(ctrl, raw_action, cfg):
    """
    Convert raw VLA action to executable Surena action.

    This is the only place where task-mode behavior should differ:
      - drawer / articulation: no sticky, force gripper open
      - pick_place / combo: preserve the continuous gripper command
      - rotation authority is mode-specific but still configurable from cfg
    """
    raw = np.asarray(raw_action, dtype=float).copy()
    if raw.shape[0] < 7:
        raise ValueError(f"VLA action must have at least 7 values, got shape {raw.shape}")

    mode = cfg.get("mode", "pick_place")
    exec_action = raw.copy()

    if mode == "drawer":
        rot_fraction = float(cfg.get("rot_fraction", cfg["rot_fraction_drawer"]))
        exec_action[3:6] = rot_fraction * np.clip(exec_action[3:6], -1.0, 1.0)
        exec_action[6] = 0.0
    elif mode == "articulation":
        rot_fraction = float(cfg.get("rot_fraction", cfg["rot_fraction_articulation"]))
        exec_action[3:6] = rot_fraction * np.clip(exec_action[3:6], -1.0, 1.0)
        exec_action[6] = 0.0
    elif mode in {"pick_place", "combo"}:
        rot_fraction = float(cfg.get("rot_fraction", cfg["rot_fraction_pick_place"]))
        exec_action[3:6] = rot_fraction * np.clip(exec_action[3:6], -0.5, 0.5)
        # Hysteresis and dwell belong to the package state machine.
        # Keep the unnormalized MiniVLA gripper output continuous here.
        exec_action[6] = raw[6]
    else:
        raise ValueError(f"No action adapter for mode={mode!r}")

    dpos_cmd = np.clip(
        exec_action[:3] * ctrl.vla.pos_scale,
        -ctrl.vla.MAX_POS_DELTA,
        ctrl.vla.MAX_POS_DELTA,
    )

    return exec_action, dpos_cmd


def _blend_guidance(ctrl, raw_action, cfg, phase,
                     pos_correction_world=None,
                     rot_correction_raw=None,
                     gripper=None):
    """Blend a raw VLA action with a bounded geometric correction.

    This is the residual/blended-control primitive for grasp assist. The VLA
    action is the base signal; guidance only ever contributes a clipped,
    phase-weighted additive correction on top of it. It never replaces the
    VLA output outright (except the gripper channel, which stays fully
    deterministic by design -- see cfg['grasp_blend_alpha_*']).

    Parameters
    ----------
    raw_action : the action array *before* this guidance call (i.e. the
        current exec_action, which already carries the raw VLA signal).
    phase : one of 'hover', 'approach', 'transport', 'lower', 'close' --
        selects the per-phase alpha from cfg.
    pos_correction_world : Cartesian correction in meters (world frame), or
        None to leave action[:3] as pure VLA.
    rot_correction_raw : additive correction for action[3:6] in raw-action
        units, or None to leave orientation as pure VLA.
    gripper : if not None, overrides action[6] (deterministic grasp state).

    Returns (action, blend_info) where blend_info is a small dict suitable
    for telemetry / overlay logging.
    """
    raw = np.asarray(raw_action, dtype=float).copy()
    out = raw.copy()

    alpha_key = {
        "hover": "grasp_blend_alpha_hover",
        "approach": "grasp_blend_alpha_approach",
        "transport": "grasp_blend_alpha_transport",
        "lower": "grasp_blend_alpha_lower",
        "close": "grasp_blend_alpha_close",
    }[phase]
    alpha = float(cfg.get(alpha_key, 0.5))

    guide_pos_raw = None
    if pos_correction_world is not None:
        kp = float(cfg.get("grasp_position_guidance_gain", 6.0))
        max_corr_world = float(cfg.get("grasp_max_position_correction", 0.03))
        pos_scale = float(ctrl.vla.pos_scale)
        # Proportional correction (not fixed-step), expressed in raw-action
        # units so its magnitude is comparable to genuine VLA dx/dy/dz.
        max_corr_raw = max_corr_world / pos_scale
        guide_pos_raw = np.clip(
            (kp * np.asarray(pos_correction_world, dtype=float)) / pos_scale,
            -max_corr_raw, max_corr_raw,
        )
        out[:3] = (1.0 - alpha) * raw[:3] + alpha * guide_pos_raw
    # else: action[:3] stays pure VLA.

    guide_rot_raw = None
    if rot_correction_raw is not None:
        max_rot = float(cfg.get("grasp_max_rotation_correction", 0.30))
        guide_rot_raw = np.clip(np.asarray(rot_correction_raw, dtype=float), -max_rot, max_rot)
        out[3:6] = raw[3:6] + alpha * guide_rot_raw
    # else: action[3:6] stays pure VLA (matches prior orientation policy).

    if gripper is not None:
        out[6] = float(gripper)

    blend_info = {
        "phase": phase,
        "alpha": alpha,
        "raw_action": raw.copy(),
        "guide_pos_raw": None if guide_pos_raw is None else np.asarray(guide_pos_raw).copy(),
        "guide_rot_raw": None if guide_rot_raw is None else np.asarray(guide_rot_raw).copy(),
        "exec_action": out.copy(),
    }
    return out, blend_info


def apply_grasp_assist(ctrl, exec_action, cfg):
    """Guide approach, grasp, lift, transfer, placement, and release.

    Guidance is a bounded, phase-weighted *residual* on top of the raw VLA
    action, blended via _blend_guidance() -- it never fully overwrites
    action[:6] the way earlier revisions did. The gripper channel
    (action[6]) remains fully deterministic during active grasp/release
    state transitions, since that requires reliable binary behavior
    regardless of VLA output.

    Per-call blend telemetry is appended to cfg['_grasp_blend_log'] so the
    calling loop can pop it and forward it to the episode logger.
    """
    body_name = cfg.get("grasp_assist_body")
    if body_name is None or ctrl.sticky is None:
        return exec_action, "none"

    action = np.asarray(exec_action, dtype=float).copy()
    candidate = next(
        (row for row in ctrl.sticky.list_candidates() if row["body_name"] == body_name),
        None,
    )
    if candidate is None:
        raise RuntimeError(f"Grasp-assist body not found: {body_name!r}")

    def _log_blend(blend_info):
        cfg.setdefault("_grasp_blend_log", []).append(blend_info)

    eef_pos, _ = ctrl.get_eef_pose()
    object_pos = np.asarray(candidate["pos"], dtype=float)
    place_target_cfg = cfg.get("grasp_place_target")
    place_target = (np.asarray(place_target_cfg, dtype=float)
                    if place_target_cfg is not None else None)

    place_body = cfg.get("grasp_place_body")
    if place_body is not None:
        bid = mujoco.mj_name2id(ctrl.model, mujoco.mjtObj.mjOBJ_BODY, place_body)
        if bid < 0:
            raise RuntimeError(f"Grasp placement body not found: {place_body!r}")
        place_offset = np.asarray(cfg.get("grasp_place_offset", (0, 0, 0)), dtype=float)
        place_target = ctrl.data.xpos[bid].copy() + place_offset

    place_site = cfg.get("grasp_place_site")
    if place_site is not None:
        cached_target = cfg.get("_grasp_place_target_world")
        if cached_target is None:
            sid = mujoco.mj_name2id(ctrl.model, mujoco.mjtObj.mjOBJ_SITE, place_site)
            if sid < 0:
                raise RuntimeError(f"Grasp placement site not found: {place_site!r}")
            cached_target = ctrl.data.site_xpos[sid].copy()
            cfg["_grasp_place_target_world"] = cached_target.copy()
        place_target = np.asarray(cached_target, dtype=float)
    place_tol = float(cfg.get("grasp_place_tolerance", 0.015))

    if ctrl.sticky.attached and place_target is not None:
        cfg.pop("_grasp_close_started", None)
        cfg["_grasp_ever_attached"] = True
        lift_z = float(cfg.get(
            "grasp_transport_lift_z",
            place_target[2] + float(cfg.get("grasp_transport_clearance", 0.10)),
        ))
        xy_distance = float(np.linalg.norm(object_pos[:2] - place_target[:2]))
        if xy_distance > place_tol and object_pos[2] < lift_z - place_tol:
            waypoint = np.array([object_pos[0], object_pos[1], lift_z])
            mode = "object_lift"
        elif xy_distance > place_tol:
            waypoint = np.array([place_target[0], place_target[1], lift_z])
            mode = "object_transfer"
        elif abs(float(object_pos[2] - place_target[2])) > place_tol:
            waypoint = place_target
            mode = "object_lower"
            previous_z = cfg.get("_grasp_lower_previous_z")
            stall_epsilon = float(cfg.get("grasp_lower_stall_epsilon", 0.0005))
            if previous_z is not None and previous_z - object_pos[2] < stall_epsilon:
                cfg["_grasp_lower_stall_count"] = int(
                    cfg.get("_grasp_lower_stall_count", 0)
                ) + 1
            else:
                cfg["_grasp_lower_stall_count"] = 0
            cfg["_grasp_lower_previous_z"] = float(object_pos[2])
            if cfg["_grasp_lower_stall_count"] >= int(
                cfg.get("grasp_lower_stall_ticks", 3)
            ):
                ctrl.vla.target_pos = eef_pos.copy()
                ctrl.vla.target_so3 = ctrl.current_eef_so3()
                action, blend_info = _blend_guidance(
                    ctrl, action, cfg, phase="close",
                    pos_correction_world=np.zeros(3),
                    rot_correction_raw=np.zeros(3),
                    gripper=1.0,
                )
                _log_blend(blend_info)
                cfg["_grasp_release_started"] = True
                cfg.pop("_grasp_lower_previous_z", None)
                cfg.pop("_grasp_lower_stall_count", None)
                return action, "object_release_settled"
        else:
            ctrl.vla.target_pos = eef_pos.copy()
            ctrl.vla.target_so3 = ctrl.current_eef_so3()
            action, blend_info = _blend_guidance(
                ctrl, action, cfg, phase="close",
                pos_correction_world=np.zeros(3),
                rot_correction_raw=np.zeros(3),
                gripper=1.0,
            )
            _log_blend(blend_info)
            cfg["_grasp_release_started"] = True
            cfg.pop("_grasp_lower_previous_z", None)
            cfg.pop("_grasp_lower_stall_count", None)
            return action, "object_release"

        if mode != "object_lower":
            cfg.pop("_grasp_lower_previous_z", None)
            cfg.pop("_grasp_lower_stall_count", None)

        delta = waypoint - object_pos
        ctrl.vla.target_pos = eef_pos.copy()
        ctrl.vla.target_so3 = ctrl.current_eef_so3()
        phase = "lower" if mode == "object_lower" else "transport"
        action, blend_info = _blend_guidance(
            ctrl, action, cfg, phase=phase,
            pos_correction_world=delta,
            rot_correction_raw=None,  # orientation stays VLA-driven
            gripper=0.0,
        )
        _log_blend(blend_info)
        return action, mode

    if place_target is not None and np.linalg.norm(object_pos - place_target) <= place_tol:
        action, blend_info = _blend_guidance(
            ctrl, action, cfg, phase="close",
            pos_correction_world=np.zeros(3),
            rot_correction_raw=np.zeros(3),
            gripper=1.0,
        )
        _log_blend(blend_info)
        return action, "object_placed"

    if ctrl.sticky.attached:
        action, blend_info = _blend_guidance(
            ctrl, action, cfg, phase="close",
            pos_correction_world=None,  # position stays VLA-driven
            rot_correction_raw=None,    # orientation stays VLA-driven
            gripper=0.0,
        )
        _log_blend(blend_info)
        return action, "grasp_hold"

    # approach (not attached, not yet close)
    distance = float(candidate["dist_to_eef"])
    approach_offset = np.asarray(cfg.get("grasp_approach_offset", (0, 0, 0)), dtype=float)
    approach_point = object_pos + approach_offset
    delta = approach_point - eef_pos
    approach_distance = float(np.linalg.norm(delta))
    approach_tolerance = float(cfg.get("grasp_approach_tolerance", 0.002))
    within_attach_radius = distance <= float(ctrl.sticky.attach_distance)

    if cfg.get("_grasp_close_started", False):
        if within_attach_radius:
            ctrl.vla.target_pos = eef_pos.copy()
            ctrl.vla.target_so3 = ctrl.current_eef_so3()
            action, blend_info = _blend_guidance(
                ctrl, action, cfg, phase="close",
                pos_correction_world=delta,
                rot_correction_raw=None,
                gripper=0.0,
            )
            _log_blend(blend_info)
            return action, "object_close"
        cfg.pop("_grasp_close_started", None)

    hover_xy_tolerance = float(cfg.get("grasp_hover_xy_tolerance", 0.02))
    hover_clearance = float(cfg.get("grasp_hover_clearance", 0.10))
    hover_z = float(cfg.get("grasp_hover_z", approach_point[2] + hover_clearance))
    xy_distance = float(np.linalg.norm(approach_point[:2] - eef_pos[:2]))

    if xy_distance > hover_xy_tolerance:
        waypoint = np.array([approach_point[0], approach_point[1],
                              max(hover_z, eef_pos[2])])
        hover_delta = waypoint - eef_pos
        ctrl.vla.target_pos = eef_pos.copy()
        ctrl.vla.target_so3 = ctrl.current_eef_so3()
        action, blend_info = _blend_guidance(
            ctrl, action, cfg, phase="hover",
            pos_correction_world=hover_delta,
            rot_correction_raw=None,
            gripper=1.0,
        )
        _log_blend(blend_info)
        return action, "object_hover_approach"

    close_pose_tolerance = float(
        cfg.get("grasp_close_pose_tolerance", approach_tolerance)
    )
    grasp_pose_ready = approach_distance <= max(approach_tolerance, close_pose_tolerance)

    if within_attach_radius and grasp_pose_ready:
        cfg["_grasp_close_started"] = True
        ctrl.vla.target_pos = eef_pos.copy()
        ctrl.vla.target_so3 = ctrl.current_eef_so3()
        action, blend_info = _blend_guidance(
            ctrl, action, cfg, phase="close",
            pos_correction_world=delta,
            rot_correction_raw=None,
            gripper=0.0,
        )
        _log_blend(blend_info)
        cfg.pop("_grasp_previous_approach_distance", None)
        cfg.pop("_grasp_approach_stall_count", None)
        return action, "object_close"

    ctrl.vla.target_pos = eef_pos.copy()
    ctrl.vla.target_so3 = ctrl.current_eef_so3()
    action, blend_info = _blend_guidance(
        ctrl, action, cfg, phase="approach",
        pos_correction_world=delta,
        rot_correction_raw=None,
        gripper=1.0,
    )
    _log_blend(blend_info)
    return action, "object_approach"


def episode_overlay_lines(cfg, step, raw_action=None, exec_action=None, ctrl=None, ik_info=None):
    lines = [
        f"step={step}",
        # f"mode={cfg.get('mode')}",
        # f"preset={cfg.get('preset_name', TASK_PRESET if 'TASK_PRESET' in globals() else '')}",
    ]

    if raw_action is not None and exec_action is not None:
        lines.append(f"raw_grip={float(raw_action[6]):+.3f} exec_grip={float(exec_action[6]):+.3f}")
    if ctrl is not None and bool(cfg.get("sticky", False)):
        lines.append(f"sticky={sticky_state(ctrl)}")
    blend_log = cfg.get("_grasp_blend_log")
    if blend_log:
        last = blend_log[-1]
        lines.append(f"guide={last['phase']} alpha={last['alpha']:.2f}")
    if ik_info is not None:
        lines.append(
            f"ik={ik_info.get('ik_stage', '?')} {ik_info.get('status')} "
            f"pe={ik_info.get('pos_err', np.nan):.3f} re={ik_info.get('rot_err', np.nan):.3f} "
            f"solver={ik_info.get('solver', '?')} seed={ik_info.get('seed', '?')}"
        )
    return lines


### 6.3 Generic VLA step loop

In [ ]:
# Generic VLA loop. This is the runner core.

def run_vla_step_loop(
    env,
    ctrl,
    cfg_or_name=None,
    *,
    show_live=True,
    terminate_on_success=True,
    logger=None,
):
    cfg = resolve_episode_cfg(cfg_or_name)
    apply_episode_control_cfg(ctrl, cfg)
    configure_sticky_for_episode(env, ctrl, cfg)
    ctrl.ik.set_intentional_contact_bodies(cfg.get("interaction_body_candidates", []))
    ctrl.reset_vla()

    cfg.pop("_grasp_ever_attached", None)
    cfg.pop("_grasp_release_started", None)
    cfg.pop("_grasp_previous_approach_distance", None)
    cfg.pop("_grasp_approach_stall_count", None)
    cfg.pop("_grasp_close_started", None)
    cfg.pop("_grasp_blend_log", None)
    success_monitor = make_episode_success_monitor(env, cfg, ctrl)
    guidance_cfg = cfg.get("contact_guidance")
    contact_guidance = None
    if guidance_cfg is not None:
        if guidance_cfg.get("type") != "hinge_contact":
            raise ValueError(f"Unsupported contact guidance: {guidance_cfg!r}")
        contact_guidance = HingeContactGuidance(
            ctrl.model, ctrl.data, ctrl.prefix + "right_eef_site", guidance_cfg
        )
        print("contact guidance: hinge_contact")

    # Telemetry: config snapshots are written once per episode, right after
    # the controller/sticky/IK are in their final per-episode configuration.
    if logger is not None:
        dt, sim_freq, ctrl_freq, _ = get_loop_params(env, verbose=False)
        logger.set_loop_params(dt, sim_freq, ctrl_freq)
        logger.set_ik_config(ctrl.ik.config)
        logger.set_sticky_config(ctrl.sticky.config() if ctrl.sticky is not None else None)
        logger.set_episode_cfg(cfg)
        logger.set_extra_meta(
            mode=cfg.get("mode"),
            instruction=cfg.get("instruction"),
            bddl_file_name=cfg.get("bddl_file_name"),
        )

    print("\nRunning VLA episode")
    print("preset:", cfg.get("preset_name"))
    print("mode:", cfg.get("mode"))
    print("instruction:", repr(cfg["instruction"]))
    print("max_vla_steps:", cfg["max_vla_steps"])
    print("pos_scale:", ctrl.vla.pos_scale)
    print("rot_scale:", ctrl.vla.rot_scale)
    print("max_rot_delta:", ctrl.vla.MAX_ROT_DELTA)

    frames = []
    raw_actions = []
    exec_actions = []
    ik_log = []
    success_log = []
    sticky_log = []
    eef_log_all = []
    obj_log_all = []
    blend_log_all = []

    stop_reason = "max_steps"
    previous_ik_info = None
    success_streak = 0

    for step in range(int(cfg["max_vla_steps"])):
        if logger is not None:
            logger.begin_vla_step(step)

        frame = render_frame(env, camera_name="agentview")
        frames.append(frame)

        if not show_live_frame(
            frame,
            lines=episode_overlay_lines(cfg, step, ctrl=ctrl),
            enabled=show_live,
        ):
            print("[LIVE] ESC pressed. Stopping.")
            stop_reason = "live_esc"
            break

        t_vla0 = time.time()
        raw_action = predict_vla_action(frame, cfg["instruction"])
        vla_inference_ms = 1000.0 * (time.time() - t_vla0)

        exec_action, dpos_cmd = transform_vla_action(ctrl, raw_action, cfg)
        guidance_mode = "none"
        if contact_guidance is not None:
            exec_action, guidance_mode = contact_guidance.prepare_action(
                ctrl, exec_action, previous_ik_info
            )
            dpos_cmd = np.clip(
                exec_action[:3] * ctrl.vla.pos_scale,
                -ctrl.vla.MAX_POS_DELTA, ctrl.vla.MAX_POS_DELTA,
            )

        pos_before, quat_before = ctrl.get_eef_pose()
        # Bound the running Cartesian target to the pose the robot actually
        # achieved before integrating the next VLA delta. This prevents a
        # transient IK miss from accumulating into an unreachable target.
        ctrl.vla.target_pos = clamp_target_lag(
            ctrl.vla.target_pos, pos_before, float(cfg["max_target_lag"])
        )
        cfg["_grasp_blend_log"] = []
        exec_action, grasp_mode = apply_grasp_assist(ctrl, exec_action, cfg)
        step_blend_records = cfg.pop("_grasp_blend_log", [])
        if step_blend_records:
            blend_log_all.append(step_blend_records[-1])
        if grasp_mode != "none":
            guidance_mode = grasp_mode
        dpos_cmd = np.clip(
            exec_action[:3] * ctrl.vla.pos_scale,
            -ctrl.vla.MAX_POS_DELTA, ctrl.vla.MAX_POS_DELTA,
        )

        raw_actions.append(raw_action.copy())
        exec_actions.append(exec_action.copy())

        use_sticky = bool(cfg.get("sticky", False))

        ik_info = ctrl.apply_vla_action(
            exec_action,
            verbose=False,
            update_sticky=use_sticky,
            max_reach=float(cfg.get("ik_max_reach", IK_MAX_REACH)),
        )
        ik_log.append(ik_info)
        previous_ik_info = ik_info

        # The VLA bridge's own running Cartesian target — this is what the
        # IK solve above was aiming for, independent of where the arm
        # actually ends up. Captured here so the logger can separate
        # IK-residual error (ik_info["pos_err"]) from servo-tracking error
        # (measured EEF vs. this commanded target, after physics settle).
        commanded_target_pos = ctrl.vla.target_pos.copy()
        commanded_target_quat = getattr(ctrl.vla.target_so3, "wxyz", None)

        if ik_info["pos_err"] > float(cfg["soft_ik_warn_err"]):
            print(
                f"[IK WARN] step={step} | "
                f"pe={ik_info['pos_err']:.4f} | status={ik_info['status']}"
            )

        if ik_info["pos_err"] > float(cfg["hard_ik_stop_err"]):
            print(
                f"[HARD STOP] step={step} | "
                f"pe={ik_info['pos_err']:.4f} | status={ik_info['status']}"
            )
            stop_reason = "hard_ik_stop"
            break

        gripper_action = float(exec_action[6]) if use_sticky else None
        t_exec0 = time.time()
        step_out = execute_joint_target(
            env,
            ctrl,
            ik_info["q_goal"],
            ctrl_ticks=int(cfg["ctrl_ticks_per_vla_action"]),
            record=True,
            render_stride=int(cfg["render_stride"]),
            camera_name="agentview",
            gripper_action=None,  # apply_vla_action already updates state once
            on_tick=(logger.on_tick if logger is not None else None),
        )
        execution_wall_ms = 1000.0 * (time.time() - t_exec0)

        step_frames = step_out[0]
        eef_log = step_out[1]
        obj_log = step_out[2] if use_sticky and len(step_out) > 2 else None

        frames.extend(step_frames)
        eef_log_all.extend(list(eef_log))
        if obj_log is not None:
            obj_log_all.extend(list(obj_log))

        if logger is not None:
            # Reuses the frames execute_joint_target already rendered at
            # render_stride cadence — no extra rendering cost.
            logger.log_video_frames(step_frames, vla_step_idx=step)

        live_stop = False
        for f in step_frames:
            if not show_live_frame(
                f,
                lines=episode_overlay_lines(cfg, step, raw_action, exec_action, ctrl, ik_info),
                enabled=show_live,
            ):
                print("[LIVE] ESC pressed. Stopping.")
                live_stop = True
                break
        if live_stop:
            stop_reason = "live_esc"
            break

        pos_after, quat_after = ctrl.get_eef_pose()
        dpos_act = pos_after - pos_before
        if contact_guidance is not None:
            contact_guidance.observe_execution()

        success, success_info = success_monitor(
            verbose=(step % int(cfg["verbose_every"]) == 0)
        )
        success_log.append(success_info)
        success_streak = success_streak + 1 if success else 0
        sticky_status_now = ctrl.sticky.status() if ctrl.sticky is not None else {"attached": False, "body_name": None}
        sticky_log.append(sticky_status_now)

        if logger is not None:
            last_blend = step_blend_records[-1] if step_blend_records else None
            logger.log_vla_step(
                keyframe=frame,
                raw_action=raw_action,
                exec_action=exec_action,
                ik_info=ik_info,
                eef_pos_before=pos_before,
                eef_quat_before=quat_before,
                eef_pos_after=pos_after,
                eef_quat_after=quat_after,
                vla_inference_ms=vla_inference_ms,
                execution_wall_ms=execution_wall_ms,
                sticky_status=sticky_status_now,
                success=bool(success),
                native_success=bool(success_info.get("native_success", False)),
                scalar_success=bool(success_info.get("scalar_success", False)),
                commanded_target_pos=commanded_target_pos,
                commanded_target_quat=commanded_target_quat,
                guidance_phase=(last_blend["phase"] if last_blend is not None else guidance_mode),
                guidance_alpha=(last_blend["alpha"] if last_blend is not None else None),
            )

        if step % int(cfg["verbose_every"]) == 0:
            blend_note = ""
            if step_blend_records:
                lb = step_blend_records[-1]
                blend_note = f" | blend={lb['phase']}:alpha={lb['alpha']:.2f}"
            print(
                f"step {step:03d} | "
                f"dpos_cmd={np.round(dpos_cmd, 4)} | "
                f"dpos_act={np.round(dpos_act, 4)} | "
                f"guide={guidance_mode}{blend_note} | "
                f"raw_grip={raw_action[6]:+.3f} | "
                f"exec_grip={exec_action[6]:+.3f} | "
                f"sticky={sticky_state(ctrl)} | "
                f"ik={ik_info.get('ik_stage')}:{ik_info['status']} | "
                f"solver={ik_info.get('solver')} | seed={ik_info.get('seed')} | "
                f"score={ik_info.get('score', np.nan):.3f} | dq={ik_info.get('joint_displacement', np.nan):.3f} | "
                f"pe={ik_info['pos_err']:.4f} | "
                f"re={ik_info['rot_err']:.4f} | "
                f"success={success}"
            )

        success_hold_steps = max(1, int(cfg.get("success_hold_steps", 1)))
        if success and success_streak < success_hold_steps:
            print(
                f"[SUCCESS PENDING] {success_streak}/{success_hold_steps} "
                f"consecutive checks | preset={cfg.get('preset_name')}"
            )
        if terminate_on_success and success_streak >= success_hold_steps:
            print(f"[SUCCESS] step={step} | preset={cfg.get('preset_name')} | mode={cfg.get('mode')}")
            stop_reason = "success"
            break

    return {
        "preset_name": cfg.get("preset_name"),
        "mode": cfg.get("mode"),
        "instruction": cfg.get("instruction"),
        "cfg": cfg,
        "frames": frames,
        "raw_actions": np.asarray(raw_actions),
        "exec_actions": np.asarray(exec_actions),
        "ik_log": ik_log,
        "success_log": success_log,
        "sticky_log": sticky_log,
        "eef_log": np.asarray(eef_log_all),
        "obj_log": np.asarray(obj_log_all) if obj_log_all else None,
        "blend_log": blend_log_all,
        "stop_reason": stop_reason,
        "ctrl": ctrl,
    }


### 6.4 Public runner entry point

In [ ]:
# Public entry point for all task modes.

def run_surena_vla_episode(
    env,
    ctrl,
    cfg_or_name=None,
    *,
    show_live=True,
    terminate_on_success=True,
    logger=None,
):
    """
    Execute one Surena × LIBERO VLA episode.

    Use this for drawer, articulation, and pick-place tasks.
    To add a new runner behavior, add a tiny branch in transform_vla_action()
    or a richer monitor in make_episode_success_monitor(); do not fork this loop.

    ``logger``: optional ``surena_vla.telemetry.EpisodeLogger``. When given,
    the full episode is recorded (config, per-step, per-tick, keyframes,
    video-cadence frames); when None, this call behaves exactly as before.
    """
    cfg_local = resolve_episode_cfg(cfg_or_name)

    if cfg_local.get("mode") == "drawer_sequence":
        raise NotImplementedError(
            "drawer_sequence should be expressed as multiple normal presets/subtasks. "
            "The current surena_manipulation catalog exposes single-task presets."
        )

    return run_vla_step_loop(
        env,
        ctrl,
        cfg_local,
        show_live=show_live,
        terminate_on_success=terminate_on_success,
        logger=logger,
    )


## 7. Full VLA episode

In [ ]:
# Full VLA episode execution cell
# Assumes: setup, VLA loading, environment creation, controller preparation, and runner cells are already run.

SHOW_LIVE = True
TERMINATE_ON_SUCCESS = True

if "env" not in globals():
    raise RuntimeError("Create the LIBERO env manually before running this cell.")
if "TASK_PRESET" not in globals():
    raise RuntimeError("Set TASK_PRESET before running this cell.")
if "cfg" not in globals():
    cfg = sm.get_preset(TASK_PRESET)
if "SESSION_META" not in globals():
    raise RuntimeError("Run the 'Load VLA model' cell first (it builds SESSION_META).")

cfg = resolve_episode_cfg(cfg)

print("Running preset:", TASK_PRESET)
print("Instruction:", cfg["instruction"])
print("Mode:", cfg["mode"])
print("Expected env class:", cfg.get("env_class"))

obs, ctrl = prepare_surena_controller(
    env,
    ctrl=globals().get("ctrl", None),
    pos_scale=cfg["pos_scale"],
    rot_scale=cfg["rot_scale"],
    max_rot_delta=cfg["max_rot_delta"],
    sticky=False,  # runner enables sticky only after it resets VLA and reads cfg
    sticky_attach_distance=cfg.get("sticky_attach_distance", HONEST_STICKY_ATTACH_DISTANCE),
    verbose=False,
)

# One compact .h5 file for this rollout: runs/<preset_name>/<preset_name>__<episode_id>.h5
episode_logger = EpisodeLogger(
    RUNS_ROOT / TASK_PRESET,
    preset_name=TASK_PRESET,
    session_meta=SESSION_META,
)
mem_start_for_log = sample_memory_usage()

t0 = time.time()
episode = None
stop_reason_for_log = "exception"
success_for_log = False

try:
    episode = run_surena_vla_episode(
        env,
        ctrl,
        cfg,
        show_live=SHOW_LIVE,
        terminate_on_success=TERMINATE_ON_SUCCESS,
        logger=episode_logger,
    )
    stop_reason_for_log = episode["stop_reason"]
    success_for_log = bool(episode["success_log"][-1].get("success", False)) if episode.get("success_log") else False
finally:
    close_live_viewer()
    # Runs whether the episode finished cleanly or raised: telemetry captured
    # so far is always written, so a crash mid-rollout doesn't lose the run.
    mem_end_for_log = sample_memory_usage()
    episode_logger.finalize(
        success=success_for_log,
        stop_reason=stop_reason_for_log,
        completed=(episode is not None),
        mem_start=mem_start_for_log,
        mem_end=mem_end_for_log,
    )
    episode_log_path = episode_logger.save()
    if episode is not None:
        # Keep the rollout identity on the in-memory episode so all later
        # artifacts can use exactly the same preset and episode ID.
        episode["episode_id"] = episode_logger.episode_id
        episode["telemetry_path"] = episode_log_path
    print("Telemetry saved:", episode_log_path)

elapsed = time.time() - t0

print("\nEpisode complete.")
print("preset:", TASK_PRESET)
print("mode:", episode["mode"])
print("stop_reason:", episode["stop_reason"])
print("frames:", len(episode["frames"]))
print("elapsed:", f"{elapsed:.1f} s")

if episode.get("ik_log"):
    print(f"IK exact-ok rate: {100 * np.mean([r.get('ok', False) for r in episode['ik_log']]):.1f}%")
    print(f"Mean pos err: {np.mean([r['pos_err'] for r in episode['ik_log']]):.6f} m")
    print(f"Max pos err:  {np.max([r['pos_err'] for r in episode['ik_log']]):.6f} m")

if episode.get("success_log"):
    print("final success:", bool(episode["success_log"][-1].get("success", False)))

# display(make_rgb_animation(episode["frames"], fps=20, title=TASK_PRESET, max_frames=400))

## 8. Save video

In [ ]:
# Save the latest episode video beside a matching per-preset directory:
# videos/<preset>/<preset>__<episode_id>.mp4
# runs/<preset>/<preset>__<episode_id>.h5
if "episode" not in globals() or episode is None:
    raise RuntimeError("Run the full VLA episode cell first.")
if len(episode["frames"]) == 0:
    raise RuntimeError("Episode has no frames to save.")
if not episode.get("episode_id") or not episode.get("telemetry_path"):
    raise RuntimeError("Episode has no telemetry identity. Rerun the full VLA episode cell.")

telemetry_path = Path(episode["telemetry_path"])
preset_name = str(episode["preset_name"])
episode_id = str(episode["episode_id"])
artifact_stem = f"{preset_name}__{episode_id}"
if telemetry_path.stem != artifact_stem:
    raise RuntimeError(
        f"Telemetry identity mismatch: expected {artifact_stem!r}, "
        f"got {telemetry_path.stem!r}"
    )

video_path = save_episode_video(
    frames=episode["frames"],
    out_path=VIDEOS_ROOT / preset_name / f"{artifact_stem}.mp4",
    fps=60,
)
print("Matching telemetry:", telemetry_path)

In [ ]:
# Cell — Visualize Episode Like OpenVLA Action-History Demo

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def show_episode_action_history(
    episode,
    task_name=None,
    *,
    action_kind="exec",     # "exec" or "raw"
    fps=8,
    max_steps=None,
    use_fixed_colors=True,
):


    frames = episode.get("frames", [])
    if len(frames) == 0:
        raise ValueError("episode['frames'] is empty.")

    if action_kind == "raw":
        actions = np.asarray(episode.get("raw_actions", []), dtype=float)
        action_title = "Raw Action History"
    elif action_kind == "exec":
        actions = np.asarray(episode.get("exec_actions", []), dtype=float)
        action_title = "Action History"
    else:
        raise ValueError("action_kind must be 'raw' or 'exec'.")

    if actions.ndim != 2 or actions.shape[1] < 7:
        raise ValueError(f"Expected action array with shape (N, 7+), got {actions.shape}")

    actions = actions[:, :7]

    n_steps = len(actions)
    if max_steps is not None:
        n_steps = min(n_steps, int(max_steps))
        actions = actions[:n_steps]

    if n_steps <= 0:
        raise ValueError("No actions to animate.")

    # Map VLA action steps to representative rendered frames.
    # Your episode usually has thousands of frames but tens of VLA decisions.
    frame_ids = np.linspace(0, len(frames) - 1, n_steps, dtype=int)
    step_frames = [frames[i] for i in frame_ids]

    labels = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]

    # Same kind of colors as your example image.
    colors = [
        "#e74c3c",  # dx
        "#2ecc71",  # dy
        "#3498db",  # dz
        "#f39c12",  # droll
        "#9b59b6",  # dpitch
        "#1abc9c",  # dyaw
        "#34495e",  # gripper
    ]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ------------------------------------------------------------------
    # Left panel: simulation frame
    # ------------------------------------------------------------------
    im = axes[0].imshow(step_frames[0])
    axes[0].set_title(f'"{task_name or episode.get("instruction", episode.get("preset_name", "episode"))}"')
    axes[0].axis("off")

    step_text = axes[0].text(
        5,
        15,
        "",
        color="white",
        fontsize=10,
        bbox=dict(facecolor="black", alpha=0.55),
    )

    # ------------------------------------------------------------------
    # Right panel: action history
    # ------------------------------------------------------------------
    lines = []
    for j, lab in enumerate(labels):
        if use_fixed_colors:
            line, = axes[1].plot(
                [],
                [],
                label=lab,
                color=colors[j],
                linewidth=1.5,
            )
        else:
            line, = axes[1].plot(
                [],
                [],
                label=lab,
                linewidth=1.5,
            )
        lines.append(line)

    y_min = float(np.nanmin(actions))
    y_max = float(np.nanmax(actions))
    y_pad = max(0.1, 0.05 * (y_max - y_min + 1e-9))

    axes[1].set_xlim(0, max(1, n_steps - 1))
    axes[1].set_ylim(y_min - y_pad, y_max + y_pad)
    axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
    axes[1].legend(fontsize=7, loc="upper right")
    axes[1].set_title(action_title)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Value")

    def update(step_idx):
        im.set_data(step_frames[step_idx])

        step_text.set_text(
            f"Step {step_idx + 1}/{n_steps}"
        )

        x = np.arange(step_idx + 1)

        for j, line in enumerate(lines):
            line.set_data(x, actions[:step_idx + 1, j])

        return [im, step_text] + lines

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n_steps,
        interval=int(1000 / fps),
        blit=True,
    )

    plt.tight_layout()
    plt.close(fig)

    return HTML(anim.to_jshtml())


display(show_episode_action_history(
    episode,
    task_name=cfg.get("instruction", TASK_PRESET),
    action_kind="exec",   # use "raw" if you want model output before notebook modifications
    fps=8,
))

## 9. Cleanup

In [ ]:
# Cleanup
try:
    close_live_viewer()
except Exception:
    pass

try:
    env.close()
    print("Environment closed.")
except Exception as e:
    print("No environment to close or close failed:", e)

# 10. h5py Exploration

In [ ]:
import h5py, numpy as np

path = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/close_top_drawer/close_top_drawer__d2168e3d3535.h5"
with h5py.File(path, "r") as f:
    raw = f["vla_steps"]["raw_action"][:]
    ik_stage = f["vla_steps"]["ik_stage"].asstr()[:]
    ik_pos_err = f["vla_steps"]["ik_pos_err"][:]

print("step-to-step change in raw_action[:3] (near-zero = VLA output not moving = stuck signature):")
print(np.round(np.diff(raw[:, :3], axis=0), 4))
print("\nik_stage per step:", list(ik_stage))
print("ik_pos_err per step:", np.round(ik_pos_err, 4))

In [ ]:
import h5py, numpy as np
path = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/close_top_drawer/close_top_drawer__d2168e3d3535.h5"
with h5py.File(path, "r") as f:
    print("ik_collision_cost per step:", np.round(f["vla_steps"]["ik_collision_cost"][:], 4))
    print("ik_escalation_reason:", list(f["vla_steps"]["ik_escalation_reason"].asstr()[:]))
    print("tick collision_min_dist (min per tick):", np.nanmin(f["ticks"]["collision_min_dist"][:]))
    print("tick collision_hard any:", bool(np.any(f["ticks"]["collision_hard"][:])))

In [ ]:
import h5py, numpy as np
path = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/put_black_bowl_on_plate/put_black_bowl_on_plate__e4d60282b373.h5"
with h5py.File(path, "r") as f:
    print(list(f["vla_steps"]["ik_escalation_reason"].asstr()[12:21]))
    print(np.round(f["vla_steps"]["ik_collision_cost"][12:21], 4))
    print(np.nanmin(f["ticks"]["collision_min_dist"][:]))
    print(bool(np.any(f["ticks"]["collision_hard"][:])))

In [ ]:
import h5py, numpy as np
path = "/home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/runs/put_black_bowl_on_plate/put_black_bowl_on_plate__2be55c199c94.h5"
with h5py.File(path, "r") as f:
    stage = f["vla_steps"]["ik_stage"].asstr()[:]
    esc = f["vla_steps"]["ik_escalation_reason"].asstr()[:]
    ccost = f["vla_steps"]["ik_collision_cost"][:]
    print(list(zip(range(len(stage)), stage, esc, np.round(ccost,4))))
    print("tick collision_min_dist:", np.nanmin(f["ticks"]["collision_min_dist"][:]))
    print("tick collision_hard any:", bool(np.any(f["ticks"]["collision_hard"][:])))